# Alpha Text

In the blend chapter this notebook generates the figure showing white-stroked text drawn over Claude Monet's _Woman with a Parasol_ at alpha 0, 0.25, 0.5, 0.75, and 1. The text is set in Libertinus Serif, the font the book is typeset in. It is drawn opaque with a white stroke, then the whole text layer is given the alpha.

In [ ]:
import os
from pathlib import Path

import brushcue

def find_repo_root():
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / "writing/graphics/fonts").is_dir():
            return directory
    raise FileNotFoundError("Could not find the graphics book directory")

# BrushCue reads files relative to the working directory, so run from the
# monorepo root to load the book font stored in the repository.
os.chdir(find_repo_root())
ctx = brushcue.Context()

# The same font the book is typeset in.
BOOK_FONT = brushcue.Font.from_path("writing/graphics/fonts/LibertinusSerif-Regular.otf")
OUTPUT_DIRECTORY = Path("writing/graphics/chapters/blend/assets/alpha-text")

ALPHAS = [0, 0.25, 0.5, 0.75, 1]
IMAGE_SCALE = 0.3
TEXT_SIZE = 150
LINE_GAP = 20
TEXT_RGB = (0, 0, 0)
STROKE_RGB = (1, 1, 1)
STROKE_RADIUS = 3

In [ ]:
def styled_string(text, size, origin, render_style):
    @brushcue.brushcue_fn
    def style_character(character):
        return brushcue.StyledCharacter.new(character, BOOK_FONT, size, render_style)

    characters = brushcue.String(brushcue.input_parsers.parse_string_graph(text)).to_character_list()
    return brushcue.StyledString.styled_string_from_characters_free(
        characters.map(style_character),
        brushcue.Point2f.from_components(*origin),
    )

def text_bounds(text):
    """Ink bounds of the text when laid out at the origin."""
    style = brushcue.RenderStyle.fill_only(
        brushcue.Fill.solid(brushcue.ProfiledColor.from_rgba_srgb(brushcue.RGBAColor.from_components(0, 0, 0, 1)))
    )
    painter = brushcue.Painter.new().add_styled_text(styled_string(text, TEXT_SIZE, (0, 0), style))
    return brushcue.Composition.painter(painter).bounds().execute(ctx)

# Scale the painting down so the figures stay a reasonable size for the book.
painting = brushcue.Composition.monet_women_with_parasol().transform(
    brushcue.Transform2.identity().scale(brushcue.Vector2f.from_components(IMAGE_SCALE, IMAGE_SCALE))
)
painting_bounds = painting.bounds().execute(ctx)
center_x = painting_bounds.x + painting_bounds.width / 2
center_y = painting_bounds.y + painting_bounds.height / 2

In [ ]:
from IPython.display import Image, display

def solid(rgb):
    return brushcue.ProfiledColor.from_rgba_srgb(brushcue.RGBAColor.from_components(*rgb, 1))

STROKE_STYLE = brushcue.RenderStyle.brush_only(brushcue.Brush.solid(solid(STROKE_RGB), STROKE_RADIUS))
FILL_STYLE = brushcue.RenderStyle.fill_only(brushcue.Fill.solid(solid(TEXT_RGB)))

def alpha_label(alpha):
    """Two centered lines, "Alpha" over the value, with a white stroke.

    The stroke is drawn first so the fill sits on top and only the outer half
    of the stroke shows. The text is drawn opaque, then the whole text layer is
    given the alpha, so the stroke does not show through the fill.
    """
    lines = ["Alpha", f"{alpha:g}"]
    measured = [text_bounds(line) for line in lines]
    block_height = sum(bounds.height for bounds in measured) + LINE_GAP * (len(lines) - 1)
    top = center_y - block_height / 2

    origins = []
    for bounds in measured:
        origins.append((center_x - bounds.x - bounds.width / 2, top - bounds.y))
        top += bounds.height + LINE_GAP

    painter = brushcue.Painter.new()
    for style in (STROKE_STYLE, FILL_STYLE):
        for line, origin in zip(lines, origins):
            painter = painter.add_styled_text(styled_string(line, TEXT_SIZE, origin, style))
    return brushcue.Composition.painter(painter).opacity_scales(alpha)

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
for alpha in ALPHAS:
    # Alpha blend the text over the painting. The painting is opaque, so the
    # text layer's alpha alone decides how much of the painting shows through.
    composition = alpha_label(alpha).blend_alpha(painting).color_convert(brushcue.ColorProfile.srgb())
    image_bytes = composition.execute(ctx).to_image_bytes(ctx)
    name = f"alpha_{str(alpha).replace('.', '_')}.png"
    (OUTPUT_DIRECTORY / name).write_bytes(bytes(image_bytes))
    display(Image(bytes(image_bytes)))